# retrieval-context evaluation

In [1]:
# install deps
!pip install sentence-transformers numpy scikit-learn pandas tqdm --quiet

In [2]:
# imports
# json/os/re handle files and text; numpy and pandas do math and tables
import json, os, re
import numpy as np
import pandas as pd
from pathlib import Path
from collections import defaultdict
# torch and sentence_transformers run the embedding models; sklearn/scipy give metrics and stats
import torch
from sentence_transformers import SentenceTransformer
from sklearn.metrics import f1_score as f1_score_fn
from sklearn.metrics import roc_auc_score
from scipy.stats import wilcoxon

import zipfile

In [ ]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# find the folder holding the dataset: kaggle input if present, else the local data folder
DATA_DIR = None
if os.path.exists("/kaggle/input"):
    for dp, _, fns in os.walk("/kaggle/input"):
        if "wb_contracts.json" in fns:
            DATA_DIR = Path(dp); break
if DATA_DIR is None:
    DATA_DIR = Path("data")
OUT = Path("/kaggle/working/data") if os.path.exists("/kaggle/working") else Path("data")
OUT.mkdir(parents=True, exist_ok=True)
def load(fn):
    with open(Path(DATA_DIR)/fn) as f: return json.load(f)

# load the json files nb1 produced
contracts   = load("wb_contracts.json")
jobs        = load("wb_jobs.json")
freelancers = load("wb_freelancers.json")
# a contract counts as relevant to a role if their skill overlap is at least 0.30.
# k_values = the cutoffs (top 1 and top 3) the metrics use
REL_THRESH = 0.30
K_VALUES = (1, 3)
print("device", DEVICE, "| contracts", len(contracts), "| jobs", len(jobs), "| freelancers", len(freelancers))

device cuda | contracts 634 | jobs 900 | freelancers 400


## lookups, skill vocab and builders

In [4]:
# lookups and the text builders for role, contract, portfolio
# quick lookups: job by id, and role by id together with its required skills as a set
job_by_id = {j["job_post_id"]: j for j in jobs}
role_by_id = {}
for j in jobs:
    for r in j["roles"]:
        req = set(s.lower() for s in (r.get("required_skills") or []))
        role_by_id[r["job_role_id"]] = (j, r, req)
# each freelancer's skills as a lowercase set for fast matching
fl_skills = {f["freelancer_id"]: set(s.lower() for s in (f.get("skills") or {}).keys()) for f in freelancers}

# gather every skill word we know so we can scan free text for skills
vocab = set()
for own in fl_skills.values(): vocab |= own
for _, _, req in role_by_id.values(): vocab |= req
# precompile a search pattern per skill (longest first) to find skills inside text
_pats = [(s, re.compile(r"\b" + re.escape(s) + r"\b", re.I)) for s in sorted(vocab, key=len, reverse=True)]

# return the known skills that appear in a piece of text
def skills_in_text(text):
    t = text or ""
    return set(s for s, p in _pats if p.search(t))

# the text we embed for a job role: title, role, skills, description
def role_text(job, role):
    p = [f"Job Title: {job['job_title']}", f"Role: {role['role_title']}"]
    if role.get("required_skills"):  p.append("Required Skills: " + ", ".join(role["required_skills"]))
    if role.get("preferred_skills"): p.append("Preferred Skills: " + ", ".join(role["preferred_skills"]))
    if job.get("job_description"):    p.append("Description: " + job["job_description"])
    return "\n".join(p)

# the text we embed for a past contract: role, job, description, rating, review
def contract_text(c):
    p = [f"Completed Role: {c['role_title']}", f"Job: {c['job_title']}"]
    if c.get("actual_completion_date"): p.append("Completed: " + str(c["actual_completion_date"]))
    jd = (job_by_id.get(c["job_post_id"], {}) or {}).get("job_description")
    # nomic handles ~8k tokens, so keep a generous 1000 chars of the description
    if jd: p.append("Description: " + jd[:1000])
    if c.get("overall_rating") is not None: p.append(f"Client Rating: {c['overall_rating']}/5")
    if c.get("review_text"): p.append("Client Review: " + c["review_text"])
    return "\n".join(p)

# the text we embed for a portfolio item: title, date, description
def portfolio_text(p):
    parts = [f"Portfolio Project: {p['project_title']}"]
    if p.get("completion_date"): parts.append("Completed: " + str(p["completion_date"]))
    if p.get("description"): parts.append("Description: " + p["description"])
    return "\n".join(parts)

# overlap of two sets: shared items divided by total items (0 to 1)
def jaccard(a, b):
    return len(a & b) / len(a | b) if a and b else 0.0

## build candidate sets per freelancer, per source

In [5]:
# build each freelancer's candidate evidence, keep those with >=2 items
contract_cand = defaultdict(list)
for c in contracts:
    # only completed contracts, and only if we still know which role they were on
    if c.get("status") == "completed" and c["job_role_id"] in role_by_id:
        contract_cand[c["freelancer_id"]].append({
            "id": c["contract_id"], "text": contract_text(c),
            "skills": role_by_id[c["job_role_id"]][2],
            "date": c.get("actual_completion_date") or ""})

# do the same for portfolio items
portfolio_cand = defaultdict(list)
for f in freelancers:
    own = fl_skills[f["freelancer_id"]]
    for i, p in enumerate(f.get("portfolio") or []):
        text = p.get("project_title", "") + " " + (p.get("description") or "")
        portfolio_cand[f["freelancer_id"]].append({
            "id": f"{f['freelancer_id']}#p{i}", "text": portfolio_text(p),
            "skills": skills_in_text(text) & own,
            "date": p.get("completion_date") or ""})

# keep only freelancers with at least 2 items, since ranking needs things to compare
contract_cand  = {k: v for k, v in contract_cand.items() if len(v) >= 2}
portfolio_cand = {k: v for k, v in portfolio_cand.items() if len(v) >= 2}
print("freelancers with >=2 contracts:", len(contract_cand))
print("freelancers with >=2 portfolio:", len(portfolio_cand))

freelancers with >=2 contracts: 166
freelancers with >=2 portfolio: 400


## embedding models

In [ ]:
MODELS = [
    {"name": "nomic-embed-text-v1.5", "id": "nomic-ai/nomic-embed-text-v1.5",
     "trust": True,  "qp": "search_query: ", "dp": "search_document: "},
    {"name": "all-MiniLM-L6-v2", "id": "sentence-transformers/all-MiniLM-L6-v2",
     "trust": False, "qp": "", "dp": ""},
    {"name": "e5-base-v2", "id": "intfloat/e5-base-v2",
     "trust": False, "qp": "query: ", "dp": "passage: "},
]
# queries are ACTIVE job roles only (the analysis targets). contracts live on PAST
role_ids = [rid for rid, (j, r, req) in role_by_id.items() if j.get("lifecycle") == "active"]
if not role_ids:  # older datasets without the active/past split -> fall back to all roles
    role_ids = list(role_by_id.keys())
r_idx = {rid: i for i, rid in enumerate(role_ids)}
# the text for each active role, in the same order as role_ids
r_texts = [role_text(*role_by_id[rid][:2]) for rid in role_ids]
print("active-role queries:", len(role_ids), "of", len(role_by_id), "total roles")

active-role queries: 312 of 900 total roles


## metrics

In [7]:
# ranking metrics: mrr, recall@k, ndcg@k, hit@1
# discounted cumulative gain: rewards putting relevant items near the top (log discount)
def dcg(g): return sum(v / np.log2(i + 2) for i, v in enumerate(g))

# given one query's ranking, compute all the metrics
def score_ranking(order, rel_bin, rel_grade, k_values):
    out = {}
    # position of the first relevant item (1-based); mrr = 1 / that position
    first = next((i + 1 for i, x in enumerate(order) if rel_bin[x]), 0)
    out["MRR"] = 1.0 / first if first else 0.0
    total = sum(rel_bin)
    for k in k_values:
        top = order[:k]
        hit = sum(rel_bin[x] for x in top)
        # recall@k: fraction of all relevant items that landed in the top k
        out[f"Recall@{k}"] = hit / total if total else 0.0
        # ndcg@k: our gain divided by the best possible gain, so 1.0 means perfect order
        ideal = sorted((rel_grade[x] for x in order), reverse=True)
        out[f"nDCG@{k}"] = dcg([rel_grade[x] for x in top]) / (dcg(ideal[:k]) or 1.0)
    # hit@1: 1 if the very top item is relevant, else 0
    out["Hit@1"] = 1.0 if rel_bin[order[0]] else 0.0
    return out

## run over both sources

In [ ]:
# rank each freelancer evidence per active role
def make_pairs(cand):
    tasks = []
    for fid, items in cand.items():
        for rid in role_ids:                      # active-role queries only
            treq = role_by_id[rid][2]
            grade = [jaccard(treq, it["skills"]) for it in items]
            binr  = [g >= REL_THRESH for g in grade]
            # skip roles where everything is relevant or nothing is (nothing to rank)
            if any(binr) and not all(binr):
                tasks.append((fid, rid, items, binr, grade))
    return tasks

# rank a freelancer's items by cosine similarity to the role, then score that ranking
def eval_embed(tasks, emb_by_id, r_vecs):
    agg = defaultdict(list)
    for fid, rid, items, binr, grade in tasks:
        rv = r_vecs[r_idx[rid]]
        # dot product of normalized vectors = cosine similarity
        sims = [float(np.dot(emb_by_id[it["id"]], rv)) for it in items]
        # sort items best (most similar) first
        order = list(np.argsort(sims)[::-1])
        for kk, vv in score_ranking(order, binr, grade, K_VALUES).items():
            agg[kk].append(vv)
    mean = {kk: float(np.mean(vv)) for kk, vv in agg.items()}
    return mean, dict(agg)

# baseline: rank by most recent date instead of embedding, to check embeddings actually help
def eval_recency(tasks):
    agg = defaultdict(list)
    for fid, rid, items, binr, grade in tasks:
        order = list(np.argsort([it["date"] for it in items])[::-1])
        for kk, vv in score_ranking(order, binr, grade, K_VALUES).items():
            agg[kk].append(vv)
    mean = {kk: float(np.mean(vv)) for kk, vv in agg.items()}
    return mean, dict(agg)

tasks = {"contract": make_pairs(contract_cand), "portfolio": make_pairs(portfolio_cand)}
print("eval pairs | contract:", len(tasks["contract"]), "| portfolio:", len(tasks["portfolio"]))

rows = {}
raw = {}
# load each model, embed roles once and evidence once, then score every task
for cfg in MODELS:
    print("encoding", cfg["name"])
    m = SentenceTransformer(cfg["id"], trust_remote_code=cfg["trust"], device=DEVICE)
    # role is the document side, evidence is the query side (nomic's asymmetric prefixing)
    R = np.asarray(m.encode([cfg["dp"] + t for t in r_texts], normalize_embeddings=True, batch_size=32, show_progress_bar=True))
    for src, cand in [("contract", contract_cand), ("portfolio", portfolio_cand)]:
        ids  = [it["id"] for its in cand.values() for it in its]
        txts = [it["text"] for its in cand.values() for it in its]
        # embed the evidence items (query side)
        V = np.asarray(m.encode([cfg["qp"] + t for t in txts], normalize_embeddings=True, batch_size=32, show_progress_bar=True))
        mean, raw_scores = eval_embed(tasks[src], dict(zip(ids, V)), R)
        rows[(cfg["name"], src)] = mean
        raw[(cfg["name"], src)] = raw_scores

for src in ("contract", "portfolio"):
    mean, raw_scores = eval_recency(tasks[src])
    rows[("recency_baseline", src)] = mean
    raw[("recency_baseline", src)] = raw_scores

eval pairs | contract: 5651 | portfolio: 2007
encoding nomic-embed-text-v1.5


modules.json:   0%|          | 0.00/255 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/140 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/58.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

Batches:   0%|          | 0/10 [00:00<?, ?it/s]

Batches:   0%|          | 0/16 [00:00<?, ?it/s]

Batches:   0%|          | 0/60 [00:00<?, ?it/s]

## compare

In [9]:
# build the comparison table and save per-source results
# the metric columns for the results table
cols = ["Hit@1", "MRR"] + [f"Recall@{k}" for k in K_VALUES] + [f"nDCG@{k}" for k in K_VALUES]
# put every model and source result into one table
df = pd.DataFrame([{"model": mdl, "source": src, **{c: round(r.get(c, 0), 4) for c in cols}}
                   for (mdl, src), r in rows.items()])

# split into a contract table and a portfolio table
df_contract = df[df["source"] == "contract"].drop(columns="source").sort_values("model").reset_index(drop=True)
df_portfolio = df[df["source"] == "portfolio"].drop(columns="source").sort_values("model").reset_index(drop=True)

print("Contract embedding metrics result:")
print(df_contract.to_string(index=False))
print()
print("Portfolio embedding metrics result:")
print(df_portfolio.to_string(index=False))

contract_json = {m: r for (m, s), r in rows.items() if s == "contract"}
portfolio_json = {m: r for (m, s), r in rows.items() if s == "portfolio"}

# save both tables as json for the thesis
with open(OUT / "wb_retrieval_context_results_contract.json", "w") as f:
    json.dump(contract_json, f, indent=2)
with open(OUT / "wb_retrieval_context_results_portfolio.json", "w") as f:
    json.dump(portfolio_json, f, indent=2)

print()
print("saved ->", OUT / "wb_retrieval_context_results_contract.json")
print("saved ->", OUT / "wb_retrieval_context_results_portfolio.json")

Contract embedding metrics result:
                model  Hit@1    MRR  Recall@1  Recall@3  nDCG@1  nDCG@3
     all-MiniLM-L6-v2 0.6142 0.7817    0.5312    0.9196  0.6966  0.8517
           e5-base-v2 0.6328 0.7923    0.5487    0.9275  0.7127  0.8625
nomic-embed-text-v1.5 0.6353 0.7941    0.5475    0.9263  0.7151  0.8627
     recency_baseline 0.4385 0.6690    0.3702    0.8695  0.5213  0.7523

Portfolio embedding metrics result:
                model  Hit@1    MRR  Recall@1  Recall@3  nDCG@1  nDCG@3
     all-MiniLM-L6-v2 0.5326 0.7266    0.2079    0.6042  0.8501  0.8895
           e5-base-v2 0.4843 0.6945    0.1928    0.5818  0.8343  0.8802
nomic-embed-text-v1.5 0.4494 0.6663    0.1854    0.5571  0.8269  0.8779
     recency_baseline 0.5590 0.7522    0.1930    0.6234  0.8459  0.8918

saved -> /kaggle/working/data/wb_retrieval_context_results_contract.json
saved -> /kaggle/working/data/wb_retrieval_context_results_portfolio.json


## significance testing

In [10]:
# wilcoxon significance test between models
# which model pairs to compare head to head
pairs_to_test = [
    ("nomic-embed-text-v1.5", "e5-base-v2"),
    ("nomic-embed-text-v1.5", "all-MiniLM-L6-v2"),
    ("e5-base-v2", "all-MiniLM-L6-v2"),
]

# wilcoxon test: is the score gap between two models real, or just random noise?
def run_sig(src):
    out_rows = []
    for m1, m2 in pairs_to_test:
        for metric in cols:
            # the per-task scores for each model on this metric
            a = np.array(raw[(m1, src)][metric])
            b = np.array(raw[(m2, src)][metric])
            diff = a - b
            # if the two models scored identically everywhere, there is nothing to test
            if np.all(diff == 0):
                stat, p = np.nan, 1.0
            else:
                stat, p = wilcoxon(a, b)
            out_rows.append({
                "model_a": m1, "model_b": m2, "metric": metric,
                "mean_diff": round(float(np.mean(diff)), 4),
                "p_value": round(float(p), 4),
                # p below 0.05 means the difference is statistically significant
                "significant_0.05": bool(p < 0.05),
            })
    return out_rows

sig_contract = run_sig("contract")
sig_portfolio = run_sig("portfolio")

print("Contract significance result:")
print(pd.DataFrame(sig_contract).to_string(index=False))
print()
print("Portfolio significance result:")
print(pd.DataFrame(sig_portfolio).to_string(index=False))

with open(OUT / "wb_retrieval_significance_results_contract.json", "w") as f:
    json.dump(sig_contract, f, indent=2)
with open(OUT / "wb_retrieval_significance_results_portfolio.json", "w") as f:
    json.dump(sig_portfolio, f, indent=2)

print()
print("saved ->", OUT / "wb_retrieval_significance_results_contract.json")
print("saved ->", OUT / "wb_retrieval_significance_results_portfolio.json")

Contract significance result:
              model_a          model_b   metric  mean_diff  p_value  significant_0.05
nomic-embed-text-v1.5       e5-base-v2    Hit@1     0.0025   0.6570             False
nomic-embed-text-v1.5       e5-base-v2      MRR     0.0018   0.5268             False
nomic-embed-text-v1.5       e5-base-v2 Recall@1    -0.0012   0.5649             False
nomic-embed-text-v1.5       e5-base-v2 Recall@3    -0.0012   0.5801             False
nomic-embed-text-v1.5       e5-base-v2   nDCG@1     0.0023   0.5751             False
nomic-embed-text-v1.5       e5-base-v2   nDCG@3     0.0001   0.8446             False
nomic-embed-text-v1.5 all-MiniLM-L6-v2    Hit@1     0.0211   0.0004              True
nomic-embed-text-v1.5 all-MiniLM-L6-v2      MRR     0.0124   0.0001              True
nomic-embed-text-v1.5 all-MiniLM-L6-v2 Recall@1     0.0163   0.0076              True
nomic-embed-text-v1.5 all-MiniLM-L6-v2 Recall@3     0.0067   0.0122              True
nomic-embed-text-v1.5 al

## relevance threshold tuning (per model, contract)

In [11]:
# tune the cosine similarity cutoff for EACH embedding model
# for each model: embed roles (document side) and contract evidence (query side),
# collect (similarity, is_relevant) for every contract-vs-role pair, then sweep cutoffs
candidate_thresholds = np.round(np.arange(0.10, 0.91, 0.01), 2)
ids  = [it["id"]   for its in contract_cand.values() for it in its]
txts = [it["text"] for its in contract_cand.values() for it in its]

threshold_by_model = {}
for cfg in MODELS:
    print("tuning", cfg["name"])
    m = SentenceTransformer(cfg["id"], trust_remote_code=cfg["trust"], device=DEVICE)
    # role = document side, evidence = query side
    R_thresh = np.asarray(m.encode([cfg["dp"] + t for t in r_texts], normalize_embeddings=True, batch_size=32, show_progress_bar=True))
    V_thresh = np.asarray(m.encode([cfg["qp"] + t for t in txts],   normalize_embeddings=True, batch_size=32, show_progress_bar=True))
    emb_by_id = dict(zip(ids, V_thresh))

    # (cosine similarity, is_relevant) for every contract-vs-role pair
    scores, labels = [], []
    for fid, rid, items, binr, grade in tasks["contract"]:
        rv = R_thresh[r_idx[rid]]
        for it, b in zip(items, binr):
            scores.append(float(np.dot(emb_by_id[it["id"]], rv)))
            labels.append(int(b))
    scores = np.array(scores); labels = np.array(labels)

    # try every cutoff from 0.10 to 0.90 and keep the one with the best f1
    grid, best_t, best_f1 = [], None, -1
    for t in candidate_thresholds:
        preds = (scores >= t).astype(int)
        f1 = f1_score_fn(labels, preds, zero_division=0)  # balances precision and recall at this cutoff
        grid.append({"threshold": float(t), "f1": round(float(f1), 4)})
        if f1 > best_f1:
            best_f1, best_t = f1, t
    auc = roc_auc_score(labels, scores)  # overall separability, independent of any single cutoff

    threshold_by_model[cfg["name"]] = {
        "n_pairs": int(len(scores)),
        "best_threshold": float(best_t),
        "best_f1": round(float(best_f1), 4),
        "roc_auc": round(float(auc), 4),
        "grid": grid,
    }
    print(f"  {cfg['name']}: best_threshold={best_t} | f1={round(float(best_f1), 4)} | roc_auc={round(float(auc), 4)}")

# side-by-side summary across models
print()
print(pd.DataFrame([{"model": k, "best_threshold": v["best_threshold"], "best_f1": v["best_f1"], "roc_auc": v["roc_auc"]}
                    for k, v in threshold_by_model.items()]).to_string(index=False))

# save per-model results (best cutoff, f1, roc-auc, and the full grid for each model)
with open(OUT / "wb_relevance_threshold_tuning.json", "w") as f:
    json.dump(threshold_by_model, f, indent=2)
print("\nsaved ->", OUT / "wb_relevance_threshold_tuning.json")


tuning nomic-embed-text-v1.5


<All keys matched successfully>


Batches:   0%|          | 0/10 [00:00<?, ?it/s]

Batches:   0%|          | 0/16 [00:00<?, ?it/s]

  nomic-embed-text-v1.5: best_threshold=0.63 | f1=0.6148 | roc_auc=0.6935
tuning all-MiniLM-L6-v2


...

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: intfloat/e5-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/10 [00:00<?, ?it/s]

Batches:   0%|          | 0/16 [00:00<?, ?it/s]

  e5-base-v2: best_threshold=0.78 | f1=0.6058 | roc_auc=0.6853

                model  best_threshold  best_f1  roc_auc
nomic-embed-text-v1.5            0.63   0.6148   0.6935
     all-MiniLM-L6-v2            0.30   0.5909   0.6576
           e5-base-v2            0.78   0.6058   0.6853

saved -> /kaggle/working/data/wb_relevance_threshold_tuning.json


## zip results

In [12]:
# zip only the json result files this notebook made. we list them explicitly and skip
# any .zip (and the archive itself), so the zip can never end up nested inside itself.
result_files = [
    "wb_retrieval_context_results_contract.json",
    "wb_retrieval_context_results_portfolio.json",
    "wb_retrieval_significance_results_contract.json",
    "wb_retrieval_significance_results_portfolio.json",
    "wb_relevance_threshold_tuning.json",
]

# put the zip at the working-dir top level so the data/ folder keeps only the result json
zip_path = (Path("/kaggle/working") if os.path.exists("/kaggle/working") else OUT.parent) / "wb_nb2_results.zip"
if zip_path.exists():
    zip_path.unlink()

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for fname in result_files:
        fpath = OUT / fname
        if fpath.suffix == ".zip" or fpath.resolve() == zip_path.resolve():
            continue  # never zip a zip or the archive itself
        if fpath.exists():
            zf.write(fpath, arcname=fname)
            print("added ->", fname)
        else:
            print("skip (not found) ->", fname)

print("saved ->", str(zip_path))


added -> wb_retrieval_context_results_contract.json
added -> wb_retrieval_context_results_portfolio.json
added -> wb_retrieval_significance_results_contract.json
added -> wb_retrieval_significance_results_portfolio.json
added -> wb_relevance_threshold_tuning.json
saved -> /kaggle/working/wb_nb2_results.zip
